<a href="https://colab.research.google.com/github/prachimishraa/GenAI/blob/main/GenAI_Lab_3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U keras-tuner

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from sklearn.metrics import classification_report, confusion_matrix

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(class_names)

def build_tunable_cnn(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(32, 32, 3)))
    model.add(layers.RandomFlip("horizontal"))
    model.add(layers.RandomRotation(0.1))

    for i in range(hp.Int("num_conv_blocks", min_value=2, max_value=3, step=1)):
        filters = hp.Choice(f"conv_filters_{i}", values=[32, 64, 128])

        model.add(layers.Conv2D(filters=filters, kernel_size=(3, 3), padding="same"))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation("relu"))

        model.add(layers.Conv2D(filters=filters, kernel_size=(3, 3), padding="same"))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation("relu"))

        model.add(layers.MaxPooling2D(pool_size=(2, 2)))

        dropout_rate = hp.Float(f"conv_dropout_{i}", min_value=0.1, max_value=0.3, step=0.1)
        model.add(layers.Dropout(rate=dropout_rate))

    model.add(layers.Flatten())

    dense_units = hp.Int("dense_units", min_value=128, max_value=512, step=128)
    model.add(layers.Dense(units=dense_units, activation="relu"))

    dense_dropout = hp.Float("dense_dropout", min_value=0.2, max_value=0.5, step=0.1)
    model.add(layers.Dropout(rate=dense_dropout))

    model.add(layers.Dense(num_classes, activation="softmax"))

    learning_rate = hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

tuner = kt.Hyperband(
    hypermodel=build_tunable_cnn,
    objective="val_accuracy",
    max_epochs=15,
    factor=3,
    directory="colab_cnn_tuning",
    project_name="cifar10_tuning"
)

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

tuner.search(
    X_train, y_train,
    epochs=15,
    validation_split=0.2,
    batch_size=64,
    callbacks=[early_stop]
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)

history = best_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

y_pred_probs = best_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))

Trial 1 Complete [00h 17m 59s]
val_accuracy: 0.527400016784668

Best val_accuracy So Far: 0.527400016784668
Total elapsed time: 00h 17m 59s

Search: Running Trial #2

Value             |Best Value So Far |Hyperparameter
3                 |2                 |num_conv_blocks
128               |32                |conv_filters_0
0.1               |0.1               |conv_dropout_0
64                |128               |conv_filters_1
0.1               |0.2               |conv_dropout_1
512               |256               |dense_units
0.2               |0.2               |dense_dropout
0.001             |0.0001            |learning_rate
2                 |2                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
0                 |0                 |tuner/round

Epoch 1/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3494 - loss: 1.8193